# Step 6 & 7: Cohort/Retention Analysis and Customer Segmentation (RFM)

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/ecommerce.db')


## Assign cohorts by first purchase month

In [2]:
cohort_df = pd.read_sql("""
WITH first_purchase AS (
    SELECT customer_id, MIN(strftime('%Y-%m', order_date)) AS cohort_month
    FROM orders
    GROUP BY customer_id
),
orders_with_cohort AS (
    SELECT o.customer_id, fp.cohort_month, strftime('%Y-%m', o.order_date) AS order_month
    FROM orders o
    JOIN first_purchase fp ON o.customer_id = fp.customer_id
)
SELECT cohort_month, order_month, COUNT(DISTINCT customer_id) AS active_customers
FROM orders_with_cohort
GROUP BY cohort_month, order_month
ORDER BY cohort_month, order_month
""", conn)

cohort_df.head(10)


,cohort_month,order_month,active_customers
0,2024-07,2024-07,19
1,2024-07,2024-08,3
2,2024-07,2024-09,3
3,2024-07,2024-11,3
4,2024-07,2024-12,3
5,2024-07,2025-01,4
6,2024-07,2025-02,3
7,2024-07,2025-03,3
8,2024-07,2025-04,1
9,2024-07,2025-05,2


## Retention table (cohort month x months since first purchase)

In [3]:
cohort_df['cohort_month'] = pd.to_datetime(cohort_df['cohort_month'])
cohort_df['order_month'] = pd.to_datetime(cohort_df['order_month'])

cohort_df['period_number'] = (
    (cohort_df['order_month'].dt.year - cohort_df['cohort_month'].dt.year) * 12 +
    (cohort_df['order_month'].dt.month - cohort_df['cohort_month'].dt.month)
)

retention_pivot = cohort_df.pivot_table(
    index='cohort_month', columns='period_number', values='active_customers'
)

cohort_sizes = retention_pivot[0]
retention_rate = retention_pivot.divide(cohort_sizes, axis=0).round(3)
retention_rate.head(10)


period_number,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
cohort_month,,,,,,,,,,,,,,,,,,,,,
2024-07-01,1.0,0.158,0.158,NaN,0.158,0.158,0.211,0.158,0.158,0.053,...,0.211,0.105,0.211,0.158,0.105,0.053,0.053,0.105,0.158,0.211
2024-08-01,1.0,0.136,0.182,NaN,0.227,0.091,0.227,NaN,0.091,0.136,...,0.182,0.091,0.182,0.136,0.182,0.091,0.227,0.045,NaN,NaN
2024-09-01,1.0,0.167,0.056,0.167,0.111,0.056,0.222,0.167,0.111,0.111,...,0.278,0.056,0.111,0.167,0.111,0.111,0.111,0.111,NaN,NaN
2024-10-01,1.0,0.059,0.059,0.176,0.059,0.235,0.176,0.176,0.118,0.235,...,0.176,0.176,0.059,0.118,0.176,0.118,0.176,NaN,NaN,NaN
2024-11-01,1.0,0.053,0.105,0.053,0.105,0.158,0.263,0.368,0.158,0.421,...,0.105,0.263,0.053,0.158,0.421,NaN,NaN,NaN,NaN,NaN
2024-12-01,1.0,NaN,0.286,0.286,0.143,0.143,0.071,0.071,0.357,0.143,...,0.071,0.143,0.143,0.143,0.071,NaN,NaN,NaN,NaN,NaN
2025-01-01,1.0,0.182,0.182,0.182,0.091,0.273,0.091,0.091,0.273,0.273,...,NaN,0.182,0.182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-02-01,1.0,NaN,0.091,0.091,0.273,0.182,0.182,0.182,0.091,0.091,...,0.273,0.182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-01,1.0,0.214,0.286,0.071,0.143,0.143,0.071,0.071,0.143,0.143,...,0.214,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Churned vs repeat customers

In [4]:
pd.read_sql("""
WITH order_counts AS (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY customer_id
)
SELECT
    CASE WHEN n_orders > 1 THEN 'repeat' ELSE 'one_time' END AS customer_type,
    COUNT(*) AS n_customers
FROM order_counts
GROUP BY customer_type
""", conn)


,customer_type,n_customers
0,one_time,16
1,repeat,168


## Customer segmentation - frequency and spend tier

In [5]:
cust_stats = pd.read_sql("""
SELECT c.customer_id, c.name,
       COUNT(DISTINCT o.order_id) AS n_orders,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spend,
       MAX(o.order_date) AS last_order_date
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.name
""", conn)

cust_stats.head()


,customer_id,name,n_orders,total_spend,last_order_date
0,1,Allison Hill,3,10467.74,2026-02-27
1,2,Javier Johnson,2,2517.25,2025-12-09
2,3,Kimberly Robinson,1,3348.48,2025-09-26
3,4,Daniel Gallagher,3,1213.62,2025-09-29
4,5,Monica Herrera,2,5196.01,2025-06-26


In [6]:
def freq_segment(n):
    if n == 1:
        return 'one-time'
    elif n <= 3:
        return 'occasional'
    else:
        return 'loyal'

cust_stats['frequency_segment'] = cust_stats['n_orders'].apply(freq_segment)

spend_low, spend_high = cust_stats['total_spend'].quantile([0.33, 0.66])

def spend_segment(x):
    if x <= spend_low:
        return 'low'
    elif x <= spend_high:
        return 'medium'
    else:
        return 'high'

cust_stats['spend_tier'] = cust_stats['total_spend'].apply(spend_segment)
cust_stats.head()


,customer_id,name,n_orders,total_spend,last_order_date,frequency_segment,spend_tier
0,1,Allison Hill,3,10467.74,2026-02-27,occasional,high
1,2,Javier Johnson,2,2517.25,2025-12-09,occasional,low
2,3,Kimberly Robinson,1,3348.48,2025-09-26,one-time,low
3,4,Daniel Gallagher,3,1213.62,2025-09-29,occasional,low
4,5,Monica Herrera,2,5196.01,2025-06-26,occasional,low


## RFM analysis (Recency, Frequency, Monetary)

In [7]:
cust_stats['last_order_date'] = pd.to_datetime(cust_stats['last_order_date'])
today = pd.Timestamp.today()
cust_stats['recency_days'] = (today - cust_stats['last_order_date']).dt.days

cust_stats['R_score'] = pd.qcut(cust_stats['recency_days'], 4, labels=[4, 3, 2, 1]).astype(int)
cust_stats['F_score'] = pd.qcut(cust_stats['n_orders'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
cust_stats['M_score'] = pd.qcut(cust_stats['total_spend'], 4, labels=[1, 2, 3, 4]).astype(int)

cust_stats['RFM_score'] = cust_stats['R_score'] + cust_stats['F_score'] + cust_stats['M_score']

cust_stats.sort_values('RFM_score', ascending=False).head(10)


,customer_id,name,n_orders,total_spend,last_order_date,frequency_segment,spend_tier,recency_days,R_score,F_score,M_score,RFM_score
134,147,Jeremy Coleman,6,11134.31,2026-06-01,loyal,high,41,4,4,4,12
129,140,Anthony Frye,5,11461.44,2026-06-30,loyal,high,12,4,4,4,12
160,174,Sandra Davis,9,22741.62,2026-07-06,loyal,high,6,4,4,4,12
156,170,Jeremy Mitchell,6,15689.23,2026-06-25,loyal,high,17,4,4,4,12
107,114,Laura Haney,6,12495.53,2026-07-06,loyal,high,6,4,4,4,12
114,121,Alexis Herrera,7,12776.42,2026-06-21,loyal,high,21,4,4,4,12
105,112,Julie Roberts,6,14402.23,2026-06-22,loyal,high,20,4,4,4,12
74,81,Diane Garcia,6,12996.76,2026-07-05,loyal,high,7,4,4,4,12
168,182,Paul Kelly,8,16010.53,2026-06-09,loyal,high,33,4,4,4,12
61,68,Jacob Obrien,5,11422.78,2026-07-07,loyal,high,5,4,3,4,11


In [8]:
cust_stats.to_csv('../output/sample_reports/rfm_segmentation.csv', index=False)
conn.close()
print('saved rfm report')


saved rfm report
